# Phase 07c — Grounded Multimodal RAG

**Main question:** How do we reason over retrieved text and images together?

In Phase 07b we learned to retrieve visual evidence using CLIP and fuse it with text results via RRF.
The retriever now surfaces the *right* evidence — but the generator still only sees text.

In this notebook we close the loop:
the retrieved original images are passed directly to the VLM alongside the retrieved text,
and the answer cites both forms of evidence.

**Sections:**
1. Retrieve text — build and query the text index
2. Retrieve visual evidence — CLIP retrieval, display figures
3. Fuse evidence — `MultimodalRetriever`, inspect RRF diagnostics
4. Construct multimodal context — grounded prompt, text vs. image split
5. Multimodal VLM generation — send question + evidence to the VLM
6. Grounded answer — resolve `[T#]` / `[V#]` to document/page/figure
7. Compare four systems — Text RAG, Caption RAG, CLIP+text, Full Multimodal RAG

**Package:** `mrta-rag[retrieval,multimodal]`

> **Note:** Sections 5–7 require a running Ollama VLM (e.g. `qwen2.5vl:latest`).
> Sections 1–4 work without a VLM.

In [ ]:
# Standard setup — run once
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image

SAMPLE_PDF = Path("../../tests/fixtures/sample.pdf")
assert SAMPLE_PDF.exists(), f"Sample PDF not found at {SAMPLE_PDF}"
print(f"Sample PDF: {SAMPLE_PDF.resolve()}")

---
## 07c.1 Retrieve Text

We start with the familiar text retrieval pipeline to establish a baseline.

In [ ]:
from mrta import Embedder, VectorStore, chunk_pdf, load_pdf

doc = load_pdf(SAMPLE_PDF)
chunks = chunk_pdf(doc)

embedder = Embedder()
text_store = VectorStore(embedder)
text_store.add(chunks)
print(f"Text index: {text_store.size} chunks")

QUERY = "What is the role of the attention mechanism?"

text_results = text_store.search_with_scores(QUERY, k=5)
print(f"\nQuery: '{QUERY}'")
print("Text results:")
for rank, (chunk, score) in enumerate(text_results, 1):
    print(f"  [{rank}] page={chunk.page}  score={score:.4f}  '{chunk.text[:60]}...'")

---
## 07c.2 Retrieve Visual Evidence

CLIP embeds images and text into the same space,
so a text query can retrieve figures without any VLM description.

In [ ]:
from mrta import CLIPEmbedder, VisualVectorStore, extract_figures

clip = CLIPEmbedder()
figures = extract_figures(doc)
print(f"Extracted {len(figures)} raster figures")

visual_records = [f.to_evidence_record() for f in figures]
visual_store = VisualVectorStore(clip)

if visual_records:
    visual_store.add(visual_records)
    print(f"Visual index: {visual_store.size} figures")

    visual_results = visual_store.search_with_scores(QUERY, k=5)
    print(f"\nVisual results for '{QUERY}':")
    for rank, (rec, score) in enumerate(visual_results, 1):
        print(f"  [{rank}] page={rec.page}  figure={rec.figure_index}  score={score:.4f}")

    # Display retrieved figures
    fig, axes = plt.subplots(1, min(3, len(visual_results)), figsize=(12, 4))
    if len(visual_results) == 1:
        axes = [axes]
    for ax, (rec, score) in zip(axes, visual_results[:3]):
        ax.imshow(rec.to_pil())
        ax.set_title(f"page={rec.page}  fig={rec.figure_index}\nscore={score:.4f}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No raster figures found — visual index is empty.")
    print("The rest of this notebook will proceed with text-only evidence.")

---
## 07c.3 Fuse Evidence

`MultimodalRetriever` runs text and visual retrieval in parallel and fuses the rankings with RRF.
The `retrieve_with_fusion_details()` method exposes the per-list ranks for inspection.

In [ ]:
from mrta import MultimodalRetriever

retriever = MultimodalRetriever(
    vector_store=text_store,
    visual_store=visual_store if visual_store.size > 0 else None,
    rrf_k=60,
)

fused = retriever.retrieve_with_fusion_details(QUERY, k_text=5, k_visual=5, k_final=8)

print(f"Fused evidence ({len(fused)} items):")
print(f"{'Rank':<6}{'Modality':<10}{'Page':<6}{'Figure':<8}{'RRF':>8}  {'Text rank':>10}  {'Visual rank':>11}")
print("-" * 65)
for i, fr in enumerate(fused, 1):
    t = fr.per_list_rank.get("text", "—")
    v = fr.per_list_rank.get("visual", "—")
    fig_idx = fr.record.figure_index or "—"
    print(f"{i:<6}{fr.source_modality:<10}{fr.record.page:<6}{str(fig_idx):<8}{fr.rrf_score:>8.4f}  {str(t):>10}  {str(v):>11}")

---
## 07c.4 Construct Multimodal Context

Before asking the VLM, we build the grounded prompt.

There are two distinct roles for visual content:

| Component | Purpose |
|---|---|
| Caption / structured description | **Retrieval** — fed to the text embedding model |
| Original image bytes | **Reasoning** — sent to the VLM alongside the prompt |

Captions help *find* relevant figures. The original image lets the VLM *understand* them.
Using captions as a proxy for images during generation would lose visual information.

In [ ]:
from mrta import load_prompt

evidence = retriever.retrieve(QUERY, k_text=5, k_visual=5, k_final=8)

text_ev = [e for e in evidence if e.modality == "text"]
visual_ev = [e for e in evidence if e.modality in ("image", "page")]
images_to_send = [ev.to_pil() for ev in visual_ev if ev.image_bytes is not None]

print(f"Evidence split:")
print(f"  Text chunks  : {len(text_ev)}")
print(f"  Visual records: {len(visual_ev)}  ({len(images_to_send)} with image bytes)")
print()

prompt = load_prompt(
    "multimodal_rag",
    question=QUERY,
    text_evidence=text_ev,
    visual_evidence=visual_ev,
)

# Show the first 60 lines of the rendered prompt
lines = prompt.strip().splitlines()
print("--- RENDERED PROMPT (first 60 lines) ---")
for line in lines[:60]:
    print(line)

---
## 07c.5 Multimodal VLM Generation

We now send the question, text evidence, and original figures to the VLM in a single call.

The `MultimodalRAG` class orchestrates this:

```
question
→ MultimodalRetriever.retrieve()    (text + visual, RRF-fused)
→ split into text / visual evidence
→ render multimodal_rag.j2 prompt
→ collect PIL images from records with image_bytes
→ VLMClient.generate(prompt, images)
→ MultimodalAnswer  (answer + [T#]/[V#] citations)
```

> **Requires:** Ollama running with a vision model, e.g. `ollama pull qwen2.5vl:latest`

In [ ]:
from mrta import MultimodalRAG, VLMClient

vlm_available = VLMClient.is_available()
print(f"VLM available: {vlm_available}")

if vlm_available:
    vlm = VLMClient()
    mmrag = MultimodalRAG(
        retriever=retriever,
        vlm=vlm,
        text_top_k=5,
        visual_top_k=5,
        fusion_top_k=8,
    )

    result = mmrag.ask(QUERY)

    print(f"Retrieval mode : {result.retrieval_mode}")
    print(f"Latency        : {result.latency_s:.2f}s")
    print(f"Text citations : {len(result.text_citations)}")
    print(f"Visual citations: {len(result.visual_citations)}")
    print()
    print("--- ANSWER ---")
    print(result.answer)
else:
    print("VLM not available. Install and start Ollama:")
    print("  ollama pull qwen2.5vl:latest")
    print("  ollama serve")

---
## 07c.6 Grounded Answer — Resolve Citations

The answer contains `[T1]`, `[T2]`, `[V1]`, `[V2]` etc.
Each label maps back to a structured `MultimodalCitation` that carries:
- `source` (filename)
- `page`
- `figure_index` (for visual evidence)
- `modality`

This lets a UI highlight the cited passage or display the cited figure.

In [ ]:
if vlm_available:
    print("Text citations:")
    for cit in result.text_citations:
        print(f"  {cit.label}  →  {cit.source} | page {cit.page}")

    print()
    print("Visual citations:")
    for cit in result.visual_citations:
        fig_str = f"| figure {cit.figure_index}" if cit.figure_index else ""
        print(f"  {cit.label}  →  {cit.source} | page {cit.page} {fig_str}")

    # Display cited figures
    cited_ids = {c.evidence_id for c in result.visual_citations}
    cited_records = [r for r in evidence if r.evidence_id in cited_ids and r.image_bytes]

    if cited_records:
        fig, axes = plt.subplots(1, len(cited_records), figsize=(5 * len(cited_records), 4))
        if len(cited_records) == 1:
            axes = [axes]
        for ax, rec in zip(axes, cited_records):
            # Find label for this record
            label = next(
                (c.label for c in result.visual_citations if c.evidence_id == rec.evidence_id),
                "?",
            )
            ax.imshow(rec.to_pil())
            ax.set_title(f"{label} — page {rec.page}, fig {rec.figure_index}")
            ax.axis("off")
        plt.suptitle("Cited visual evidence")
        plt.tight_layout()
        plt.show()
    else:
        print("(No visual citations with raster images to display.)")
else:
    print("(VLM not available — skipping citation display.)")

---
## 07c.7 Compare Four Systems

We compare the four systems that have been built across Phases 07–07c:

| System | Text | Caption | CLIP | VLM sees original images |
|---|:-:|:-:|:-:|:-:|
| A. Text RAG | ✓ | | | |
| B. Caption RAG | ✓ | ✓ | | |
| C. CLIP + text generation | ✓ | | ✓ | |
| D. Full Multimodal RAG | ✓ | | ✓ | ✓ |

For visual questions, system A is blind; B retrieves by description; C retrieves by image but
still generates text-only; only D sends the original image to the VLM.

In [ ]:
from mrta import LLMClient, rag_query


def compare_systems(query: str) -> None:
    print(f"Query: '{query}'")
    print("=" * 70)

    llm_available = LLMClient.is_available()

    # A — Text RAG
    if llm_available:
        llm = LLMClient()
        r = rag_query(query, text_store, llm, top_k=5)
        print(f"\n[A] Text RAG ({len(r['sources'])} chunks, {r['latency_s']:.2f}s)")
        print(r["answer"][:300])
    else:
        print("\n[A] Text RAG — LLM not available")

    # B — CLIP retrieval + text generation (no VLM sees images)
    clip_retriever = MultimodalRetriever(
        vector_store=text_store,
        visual_store=visual_store if visual_store.size > 0 else None,
    )
    clip_ev = clip_retriever.retrieve(query, k_text=5, k_visual=5, k_final=8)
    n_text = sum(1 for e in clip_ev if e.modality == "text")
    n_vis = sum(1 for e in clip_ev if e.modality != "text")
    print(f"\n[C] CLIP retrieval — {n_text} text + {n_vis} visual retrieved")
    print("    (Text generation without image passing not shown — requires LLM client.)")

    # D — Full Multimodal RAG
    if vlm_available:
        mmrag = MultimodalRAG(retriever=clip_retriever, vlm=VLMClient())
        r = mmrag.ask(query)
        print(f"\n[D] Full Multimodal RAG (mode={r.retrieval_mode}, {r.latency_s:.2f}s)")
        print(f"    {len(r.text_citations)} text cit. + {len(r.visual_citations)} visual cit.")
        print(r.answer[:400])
    else:
        print("\n[D] Full Multimodal RAG — VLM not available")

    print()

In [ ]:
compare_systems("What is the role of the attention mechanism?")
compare_systems("What architecture is shown in the figures?")

---
## Summary

| What we built | Where it lives |
|---|---|
| Grounded multimodal prompt | `multimodal_rag.j2` |
| Original image passing to VLM | `MultimodalRAG._build_prompt_and_images()` |
| `[T#]` / `[V#]` structured citations | `MultimodalCitation`, `MultimodalAnswer` |
| Graceful VLM fallback | `try/except LLMError` in `MultimodalRAG.ask()` |
| End-to-end multimodal RAG | `MultimodalRAG.ask()` |

**Key insight:**
Captions are *retrieval artifacts* — they help find relevant figures.
The VLM must receive the **original image** to reason about visual content.
Substituting a generated description for the image loses spatial layout, colour,
axis labels, and any detail the caption omits.

> The next step (Phase 08) is to apply this pipeline across the full set of
> teaching modes: Explain, Socratic tutor, Quiz, Compare, and Visual Evidence.